In [2]:
"""Sentinel-1 flood-screening v4: matched tracks, per-acquisition detection.

Cài: python -m pip install earthengine-api
Điền PROJECT rồi chạy: python danang_flood_sentinel1_v4.py
Nếu chưa xác thực, chạy riêng: python -c "import ee; ee.Authenticate()"

KHÔNG có bộ ngưỡng nào được chứng minh tối ưu khi chưa có nhãn kiểm chứng.
Đây là rule-based screening, không phải mô hình xác suất/rủi ro hoặc độ sâu.
Không tự tạo mẫu kiểm chứng từ chính các ngưỡng này.
Kết quả chính ưu tiên bằng chứng đồng thời VV và VH, có thể bỏ sót ngập.
ROI gồm cả phía nam Đà Nẵng; diện tích là toàn ROI, không phải thành phố.
Khoảng ngày kết thúc của Earth Engine là exclusive (không bao gồm ngày đó).

ĐẦU RA MỖI NĂM:
- danang_flood_YEAR.geojson: ứng viên có cả VV và VH, ngoài built-up WorldCover.
- danang_flood_YEAR_suspect.geojson: một kênh/đô thị/vùng nhỏ cần kiểm tra.
  Thuộc tính is_flood=1 chỉ là nhãn polygon; đọc evidence, không gộp hai lớp.
- danang_flood_YEAR_qa.tif: main_hit, suspect_hit, valid_obs, eligible_obs,
  flood_main, flood_suspect. Số hit/obs là số quan sát ngày-track, không phải
  số ngày ngập, mức tin cậy đã hiệu chuẩn hay xác suất. valid_obs=0: thiếu
  cặp quan sát đủ chuẩn; eligible_obs=0: không được luật phân loại đánh giá
  (thiếu quan sát hoặc bị loại bởi địa hình/nước có sẵn). Không gán là khô.
- danang_flood_YEAR_sources.csv: ảnh nguồn, track bị bỏ, cấu hình và diện tích.

Đây là hợp các vùng được phát hiện trong cửa sổ, không phải ảnh chụp đỉnh lũ.
Mặt nạ JRC dựa trên 1984-2021 và WorldCover dựa trên 2021 có thể lỗi thời.
Ngập dưới tán cây, trong đô thị, vùng bị che khuất radar có thể bị bỏ sót;
ruộng lúa, thay đổi canh tác/độ ẩm cũng có thể tạo dương tính giả.
Không tự động hiệu chỉnh ngưỡng từ các ứng viên do chính script tạo ra.
Để hiệu chỉnh: dùng nhãn độc lập cùng ngày, tách tập hiệu chỉnh và kiểm tra
theo vùng/sự kiện, báo precision/recall/F1/IoU và tỷ lệ có quan sát hợp lệ.
"""
from collections import defaultdict
from datetime import date
import json
import ee

PROJECT = None  # Ví dụ: 'your-google-cloud-project'; nên điền project đã bật EE.
DRIVE_FOLDER = 'Danang_Flood_v4'
ROI_BOUNDS = [107.85, 15.55, 108.45, 16.30]
CRS = 'EPSG:32648'
# Lưới cố định chung cho mọi năm. Pixel 10m KHÔNG có nghĩa chính xác vị trí 10m.
TRANSFORM = [10, 0, 0, 0, -10, 0]
P = dict(
    MIN_BASE_OBS=3,       # Số NGÀY có dữ liệu trước lũ, kiểm tra từng pixel.
    DROP_VV_DB=-1.5, DROP_VH_DB=-1.5,
    WATER_VV_DB=-14.0, WATER_VH_DB=-20.0,
    BASE_WATER_VV_DB=-18.0, BASE_WATER_VH_DB=-25.0,
    BASE_WATER_FRACTION=0.8,
    JRC_OCC=90,           # Mặt nước lịch sử rất thường xuyên; không phải chân lý 2025.
    SLOPE_MAX=10,         # Prior địa hình; cần kiểm tra độ nhạy.
    HAND_MAX=None,        # Không mặc định loại ngập đô thị/xa sông bằng HAND.
    SMOOTH_RADIUS_M=10,   # 0 = không lọc không gian; 10 = median nhẹ.
    MIN_PIXELS=1,        # Không mặc định xóa vùng nhỏ. Tăng sẽ bỏ sót vùng nhỏ.
    URBAN_RISE_DB=3.0,    # CHỈ là vùng nghi vấn; không đưa thẳng vào flood chính.
    MAX_ANGLE_DIFF=1.0,   # Sai khác góc tới trước/trong, độ; cần hiệu chỉnh.
    PASS=None,           # None = phân tích riêng từng track rồi tổng hợp nhị phân.
)
# Giữ cửa sổ bạn cung cấp, không khẳng định ngày đỉnh lũ đã được kiểm chứng.
EVENTS = [
    (
        2022,
        ('2022-08-20', '2022-10-06'),  # Mở rộng khoảng tìm ảnh nền
        ('2022-10-12', '2022-10-24'),
    ),
    (
        2025,
        ('2025-09-01', '2025-09-25'),
        ('2025-10-22', '2025-11-02'),
    ),
]
# Nếu MIN_BASE_OBS không đủ: chọn lại cửa sổ nền có >=3 lần chụp cùng track,
# kiểm tra nền KHÔNG đã ngập. Không tự mở rộng thời gian vì có thể đổi mùa vụ.


def grid(image):
    return ee.Image(image).reproject(crs=CRS, crsTransform=TRANSFORM)


def s1_col(roi, start, end):
    col = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(roi)
           .filterDate(start, end)
           .filter(ee.Filter.eq('instrumentMode', 'IW'))
           .filter(ee.Filter.eq('resolution_meters', 10))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')))
    if P['PASS']:
        col = col.filter(ee.Filter.eq('orbitProperties_pass', P['PASS']))
    return col.sort('system:time_start')


def inventory(col):
    # Chỉ tải metadata, không tải ảnh. Gom các slice cùng ngày/track sau đó.
    names = ['system:index', 'system:time_start', 'orbitProperties_pass',
             'relativeOrbitNumber_start', 'platform_number']
    rows = col.reduceColumns(ee.Reducer.toList(len(names)), names).get('list').getInfo()
    return [dict(zip(names, row)) for row in rows]


def group_tracks(rows):
    result = defaultdict(list)
    for row in rows:
        result[(row['orbitProperties_pass'], row['relativeOrbitNumber_start'])].append(row)
    return result


def day_from_ms(ms):
    from datetime import datetime, timezone
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).date().isoformat()


def daily_images(col, rows, roi):
    dates = sorted({day_from_ms(r['system:time_start']) for r in rows})
    images = []
    for day in dates:
        slices = col.filterDate(day, ee.Date(day).advance(1, 'day'))
        # Mỗi ngày chỉ tính một quan sát, không đếm trùng slice chồng nhau.
        raw = grid(slices.select(['VV', 'VH', 'angle']).mosaic()).clip(roi)
        valid = raw.mask().reduce(ee.Reducer.min())
        vv_vh = raw.select(['VV', 'VH'])
        if P['SMOOTH_RADIUS_M'] > 0:
            vv_vh = vv_vh.focal_median(
                radius=P['SMOOTH_RADIUS_M'], kernelType='circle', units='meters')
        # Focal filter không được mở rộng vùng quan sát hợp lệ sang chỗ thiếu ảnh.
        images.append(vv_vh.addBands(raw.select('angle')).updateMask(valid)
                      .set({'system:time_start': ee.Date(day).millis(), 'date': day}))
    return images


def auxiliary(roi):
    jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
    slope = ee.Terrain.slope(ee.Image('USGS/SRTMGL1_003'))
    terrain = slope.lt(P['SLOPE_MAX'])
    if P['HAND_MAX'] is not None:
        hand = ee.Image('MERIT/Hydro/v1_0_1').select('hnd')
        terrain = terrain.And(hand.lt(P['HAND_MAX']))
    # JRC và WorldCover là dữ liệu lịch sử. Không xem thiếu dữ liệu là đã xác minh khô.
    built = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').eq(50)
    return (grid(jrc.unmask(0).gte(P['JRC_OCC'])).clip(roi),
            grid(terrain).clip(roi), grid(built.unmask(0)).clip(roi))


def classify_track(base_images, event_images, aux):
    permanent, terrain, built = aux
    baseline = ee.ImageCollection.fromImages(base_images)
    before = grid(baseline.select(['VV', 'VH', 'angle']).median())
    n_before = grid(baseline.select('VV').count())
    base_water = grid(baseline.map(lambda im: ee.Image(im).select('VV')
        .lt(P['BASE_WATER_VV_DB']).And(ee.Image(im).select('VH')
        .lt(P['BASE_WATER_VH_DB'])).rename('water')).mean())
    water_exclusion = permanent.Or(base_water.gte(P['BASE_WATER_FRACTION']))
    outputs = []
    for event in event_images:
        valid = (event.mask().reduce(ee.Reducer.min())
                 .And(before.mask().reduce(ee.Reducer.min()))
                 .And(n_before.gte(P['MIN_BASE_OBS']))
                 .And(event.select('angle').subtract(before.select('angle'))
                      .abs().lte(P['MAX_ANGLE_DIFF'])))
        eligible = valid.And(terrain).And(water_exclusion.Not())
        delta = event.select(['VV', 'VH']).subtract(before.select(['VV', 'VH']))
        vv = delta.select('VV').lt(P['DROP_VV_DB']).And(
            event.select('VV').lt(P['WATER_VV_DB']))
        vh = delta.select('VH').lt(P['DROP_VH_DB']).And(
            event.select('VH').lt(P['WATER_VH_DB']))
        # VV và VH cùng đáp ứng trên CÙNG lần chụp, không ghép percentile khác ngày.
        dual = vv.And(vh)
        # Mọi tín hiệu trong built-up để ở lớp nghi vấn do giới hạn SAR đô thị.
        main = dual.And(built.Not()).And(eligible)
        urban = built.And(delta.select('VV').gte(P['URBAN_RISE_DB']))
        suspect = vv.Or(vh).Or(urban).And(main.Not()).And(eligible)
        bands = [main.rename('main_hit'), suspect.rename('suspect_hit'),
                 valid.rename('valid_obs'), eligible.rename('eligible_obs')]
        out = ee.Image.cat(bands).unmask(0).toUint16()
        outputs.append(grid(out).set('date', event.get('date')))
    return outputs


def filter_components(binary):
    binary = grid(binary).selfMask()
    if P['MIN_PIXELS'] > 1:
        count = binary.connectedPixelCount(maxSize=max(100, P['MIN_PIXELS']),
                                           eightConnected=True)
        binary = binary.updateMask(count.gte(P['MIN_PIXELS']))
    return binary


def vectors(binary, roi, properties):
    return binary.rename('is_flood').toInt8().reduceToVectors(
        reducer=ee.Reducer.countEvery(), geometry=roi, crs=CRS,
        crsTransform=TRANSFORM, geometryType='polygon', eightConnected=True,
        labelProperty='is_flood', maxPixels=1e10, bestEffort=False, tileScale=4,
    ).map(lambda f: ee.Feature(f).set(properties))


def start_task(task, label):
    task.start()
    print(f'  Đã gửi {label}: task_id={task.id}')
    return task


def phat_hien_ngap(nam, truoc_range, trong_range):
    if not (date.fromisoformat(truoc_range[0]) < date.fromisoformat(truoc_range[1])
            <= date.fromisoformat(trong_range[0]) < date.fromisoformat(trong_range[1])):
        raise ValueError('Cửa sổ thời gian sai hoặc trước/trong bị chồng nhau.')
    roi = ee.Geometry.Rectangle(ROI_BOUNDS, proj='EPSG:4326', geodesic=False)
    pre = s1_col(roi, *truoc_range)
    post = s1_col(roi, *trong_range)
    pre_rows, post_rows = inventory(pre), inventory(post)
    if not pre_rows or not post_rows:
        raise ValueError(f'{nam}: không đủ ảnh trước/trong. Kiểm tra cửa sổ ngày.')
    a, b = group_tracks(pre_rows), group_tracks(post_rows)
    aux = auxiliary(roi)
    observations, manifest = [], []
    for track in sorted(b):
        pass_name, orbit = track
        baseline_rows = a.get(track, [])
        days = sorted({day_from_ms(r['system:time_start']) for r in baseline_rows})
        used = len(days) >= P['MIN_BASE_OBS']
        print(f'[{nam}] {pass_name}, relative orbit {orbit}: '
              f'{len(days)} ngày nền; dùng={used}')
        print('  Ngày trong sự kiện:', sorted({day_from_ms(r['system:time_start']) for r in b[track]}))
        for phase, rows in [('before', baseline_rows), ('event', b[track])]:
            for row in rows:
                manifest.append(ee.Feature(None, {
                    'year': nam, 'phase': phase, 'used_track': int(used),
                    'scene_id': row['system:index'],
                    'acquired_utc_ms': row['system:time_start'],
                    'date_utc': day_from_ms(row['system:time_start']),
                    'pass': pass_name, 'relative_orbit': orbit,
                    'platform': row['platform_number'],
                }))
        if not used:
            print('  BỎ track: chọn lại nền đủ ảnh, không tự giảm MIN_BASE_OBS.')
            continue
        filt = ee.Filter.And(ee.Filter.eq('orbitProperties_pass', pass_name),
                             ee.Filter.eq('relativeOrbitNumber_start', orbit))
        before = daily_images(pre.filter(filt), baseline_rows, roi)
        during = daily_images(post.filter(filt), b[track], roi)
        observations.extend(classify_track(before, during, aux))
    if not observations:
        raise ValueError(f'{nam}: không track nào đủ >= {P["MIN_BASE_OBS"]} ngày nền. '
                         'Chọn lại cửa sổ trước lũ và xác minh điều kiện khô.')
    counts = grid(ee.ImageCollection.fromImages(observations).sum()).clip(roi)
    raw_main = counts.select('main_hit').gt(0)
    main = filter_components(raw_main)
    # Nếu chủ động lọc mảng nhỏ, chuyển pixel bị lọc sang nghi vấn, không gán khô.
    removed = raw_main.And(main.unmask(0).Not())
    suspect = counts.select('suspect_hit').gt(0).Or(removed).And(main.unmask(0).Not()).selfMask()
    observed = counts.select('valid_obs').gt(0)
    eligible = counts.select('eligible_obs').gt(0)
    # Một truy vấn QA: không xuất bản đồ trống vì hoàn toàn không có quan sát đủ chuẩn.
    area_images = ee.Image.cat([
        observed.rename('observed_m2'), eligible.rename('eligible_m2'),
        main.unmask(0).rename('main_m2'), suspect.unmask(0).rename('suspect_m2')
    ]).multiply(ee.Image.pixelArea())
    areas = area_images.reduceRegion(ee.Reducer.sum(), geometry=roi, crs=CRS,
        crsTransform=TRANSFORM, maxPixels=1e10, bestEffort=False, tileScale=4).getInfo()
    if not areas.get('observed_m2'):
        raise ValueError(f'{nam}: không pixel nào đủ quan sát chuẩn; chưa xuất dữ liệu.')
    print(f'[{nam}] Diện tích km² toàn ROI:', {k: round((v or 0)/1e6, 3) for k,v in areas.items()})
    if not areas.get('main_m2'):
        print('  Không phát hiện theo luật chính; KHÔNG có nghĩa không xảy ra ngập.')
    props = dict(year=nam, method='S1_track_daily_dual_v4',
                 before_start=truoc_range[0], before_end_exclusive=truoc_range[1],
                 event_start=trong_range[0], event_end_exclusive=trong_range[1],
                 calibrated=0, params_json=json.dumps(P, sort_keys=True),
                 roi_json=json.dumps(ROI_BOUNDS), area_scope='whole_ROI',
                 interpretation='observed_event_union_not_peak_or_risk', **areas)
    tasks = []
    for binary, suffix, evidence in [(main, '', 'dual_pol_non_built_candidate'),
                                      (suspect, '_suspect', 'single_pol_or_urban_or_small')]:
        name = f'danang_flood_{nam}{suffix}'
        fc = vectors(binary, roi, dict(props, evidence=evidence))
        tasks.append(start_task(ee.batch.Export.table.toDrive(
            collection=fc, description=name+'_v4', folder=DRIVE_FOLDER,
            fileNamePrefix=name, fileFormat='GeoJSON'), name))
    # GeoTIFF: nodata=-9999 ngoài ROI; trong ROI đọc valid_obs trước khi diễn giải 0.
    qa = counts.addBands(main.unmask(0).rename('flood_main')).addBands(
        suspect.unmask(0).rename('flood_suspect')).toInt16().clip(roi).unmask(-9999)
    tasks.append(start_task(ee.batch.Export.image.toDrive(
        image=qa, description=f'danang_flood_{nam}_qa_v4', folder=DRIVE_FOLDER,
        fileNamePrefix=f'danang_flood_{nam}_qa', region=roi, crs=CRS,
        crsTransform=TRANSFORM, maxPixels=1e10, fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True, 'noData': -9999}), 'QA raster'))
    manifest_fc = ee.FeatureCollection(manifest).map(lambda f: ee.Feature(f).set(props))
    tasks.append(start_task(ee.batch.Export.table.toDrive(
        collection=manifest_fc, description=f'danang_flood_{nam}_sources_v4',
        folder=DRIVE_FOLDER, fileNamePrefix=f'danang_flood_{nam}_sources',
        fileFormat='CSV'), 'source metadata'))
    return tasks


def main():
    if PROJECT:
        ee.Initialize(project=PROJECT)
    else:
        ee.Initialize()
    tasks = []
    for year, before, during in EVENTS:
        tasks.extend(phat_hien_ngap(year, before, during))
    print('\nĐã gửi tác vụ; chưa có nghĩa export hoàn thành.')
    print('Theo dõi tab Tasks trong Earth Engine; tải GeoJSON từ Google Drive khi COMPLETED.')
    print('File chính giữ tên danang_flood_2022.geojson và danang_flood_2025.geojson.')
    for task in tasks:
        print(task.id, task.status().get('state'))


if __name__ == '__main__':
    main()



[2022] ASCENDING, relative orbit 55: 4 ngày nền; dùng=True
  Ngày trong sự kiện: ['2022-10-15']
[2022] DESCENDING, relative orbit 18: 4 ngày nền; dùng=True
  Ngày trong sự kiện: ['2022-10-12']
[2022] DESCENDING, relative orbit 120: 4 ngày nền; dùng=True
  Ngày trong sự kiện: ['2022-10-19']
[2022] Diện tích km² toàn ROI: {'eligible_m2': 2178.416, 'main_m2': 217.578, 'observed_m2': 5331.818, 'suspect_m2': 245.247}
  Đã gửi danang_flood_2022: task_id=7OTOGJFS6LKXCD6G5CFI6CMY
  Đã gửi danang_flood_2022_suspect: task_id=DTJMWD2WGSKBARQB42UDQEFN
  Đã gửi QA raster: task_id=HWWDEBJPYDI7SU6MZJMKHPV4
  Đã gửi source metadata: task_id=BU2UH5BM7W3XSGAU7BVKDD3Z
[2025] ASCENDING, relative orbit 55: 2 ngày nền; dùng=False
  Ngày trong sự kiện: ['2025-10-23']
  BỎ track: chọn lại nền đủ ảnh, không tự giảm MIN_BASE_OBS.
[2025] DESCENDING, relative orbit 18: 4 ngày nền; dùng=True
  Ngày trong sự kiện: ['2025-10-26', '2025-11-01']
[2025] DESCENDING, relative orbit 120: 4 ngày nền; dùng=True
  Ngày trong

In [2]:
from pathlib import Path
import warnings
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import MiniMap


def read_flood_geojson(path):
    gdf = gpd.read_file(path)
    if gdf.empty:
        warnings.warn(f"{path}: không có vùng ngập trong file.")
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    if gdf.crs is None:
        raise ValueError(f"{path}: thiếu CRS, cần xác minh hệ tọa độ nguồn.")
    gdf = gdf.to_crs("EPSG:4326")
    valid = gdf.geometry.notna() & ~gdf.geometry.is_empty
    if not valid.all():
        warnings.warn(f"{path}: bỏ {int((~valid).sum())} geometry null/rỗng.")
    return gdf.loc[valid].copy()


def create_improved_flood_map(
    geojson_2022_path="danang_flood_2022.geojson",
    geojson_2025_path="danang_flood_2025.geojson",
    output_html="danang_flood_map_pro.html",
):
    print("📥 Đang đọc dữ liệu GeoJSON...")
    gdf_2022 = read_flood_geojson(geojson_2022_path)
    gdf_2025 = read_flood_geojson(geojson_2025_path)
    if gdf_2022.empty and gdf_2025.empty:
        raise ValueError("Cả hai file không có geometry để hiển thị.")

    m = folium.Map(
        location=[16.0544, 108.2022],
        zoom_start=12,
        tiles="CartoDB positron",  # Nền sáng sạch sẽ, không bị lỗi 403
        control_scale=True,
    )
    layers = [
        (2022, gdf_2022, "#2980b9", "#1b4f72", 0.35),
        (2025, gdf_2025, "#e74c3c", "#922b21", 0.60),
    ]
    for year, gdf, fill, border, opacity in layers:
        if gdf.empty:
            continue
        style = {
            "fillColor": fill, "color": border,
            "weight": 0.8, "fillOpacity": opacity,
        }
        folium.GeoJson(
            # Timestamp và các giá trị không hỗ trợ JSON được chuyển thành chuỗi.
            data=gdf.to_json(default=str, na="null", drop_id=True),
            name=f"Vùng ngập quan sát {year}",
            # Giữ riêng style của từng lớp khi dùng vòng lặp.
            style_function=lambda feature, s=style: s,
            tooltip=folium.Tooltip(f"Vùng ngập quan sát {year}", sticky=True),
            smooth_factor=0,
        ).add_to(m)

    combined = gpd.GeoSeries(
        pd.concat([gdf_2022.geometry, gdf_2025.geometry], ignore_index=True),
        crs="EPSG:4326",
    )
    minx, miny, maxx, maxy = combined.total_bounds
    m.fit_bounds([[miny, minx], [maxy, maxx]])
    folium.LayerControl(collapsed=False).add_to(m)
    MiniMap(toggle_display=True).add_to(m)

    legend_html = '''
    <div style="position:fixed;bottom:30px;left:30px;width:260px;
                max-width:calc(100vw - 80px);border:2px solid #ccc;
                z-index:9999;font-size:13px;background:white;padding:12px;
                border-radius:6px;box-shadow:0 2px 5px rgba(0,0,0,0.2);">
      <b>Vùng ngập quan sát – Đà Nẵng và lân cận</b><br><br>
      <div style="margin-bottom:8px;">
        <span style="color:#2980b9;font-size:20px;">■</span> Vùng ngập 2022
      </div>
      <div>
        <span style="color:#e74c3c;font-size:20px;">■</span> Vùng ngập 2025
      </div>
      <hr>
      <small>Kết quả phát hiện từ ảnh vệ tinh, cần kiểm chứng.
      Màu không biểu thị độ sâu hoặc mức rủi ro.
      Ngoài vùng hiển thị không đồng nghĩa không ngập.</small>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    output = Path(output_html)
    if output.resolve() in {Path(geojson_2022_path).resolve(), Path(geojson_2025_path).resolve()}:
        raise ValueError("Đường dẫn HTML không được trùng với file nguồn.")
    output.parent.mkdir(parents=True, exist_ok=True)
    m.save(str(output))
    print(f"✅ Đã tạo bản đồ: {output.resolve()}")
    return m


if __name__ == "__main__":
    create_improved_flood_map()



📥 Đang đọc dữ liệu GeoJSON...


C:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


✅ Đã tạo bản đồ: C:\Users\ACER\OneDrive\Desktop\Data Science\notebooks\danang_flood_map_pro.html
